# Validación de archivos y muestras

In [155]:
%pip install pillow pandas

In [158]:
from pathlib import Path
from PIL import Image, ImageFile
import pandas as pd

# Cambiar por la carpeta que contiene sus fuentes.
ROOT = Path("plague-identification-model/data/raw")
REPORTS = Path("reports")
REPORTS.mkdir(parents=True, exist_ok=True)

# No permitir que Pillow ignore archivos truncados.
ImageFile.LOAD_TRUNCATED_IMAGES = False

extensions = {".jpg", ".jpeg", ".png"}
paths = sorted(
    path for path in ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in extensions
)

if not paths:
    raise ValueError(f"No se encontraron imágenes en: {ROOT.resolve()}")

records = []

for path in paths:
    record = {
        "image_path": str(path),
        "validation_status": "error",
        "validation_error": "",
        "width": None,
        "height": None,
        "mode": None,
    }

    try:
        # Comprobación de integridad disponible para el formato.
        with Image.open(path) as image:
            image.verify()

        # Reabrir y forzar la decodificación de los píxeles.
        with Image.open(path) as image:
            image.load()

            record["width"] = image.width
            record["height"] = image.height
            record["mode"] = image.mode

        record["validation_status"] = "ok"

    except Exception as error:
        record["validation_error"] = (
            f"{type(error).__name__}: {error}"
        )

    records.append(record)

validation_df = pd.DataFrame(records)

validation_df.to_csv(
    REPORTS / "image_validation.csv",
    index=False
)

errors_df = validation_df[
    validation_df["validation_status"] == "error"
]

print("Imágenes revisadas:", len(validation_df))
print(
    "Imágenes que pasaron:",
    (validation_df["validation_status"] == "ok").sum()
)
print("Archivos con errores:", len(errors_df))

display(errors_df)

Imágenes revisadas: 6338
Imágenes que pasaron: 6338
Archivos con errores: 0


,image_path,validation_status,validation_error,width,height,mode


# Revisión de duplicados


| Tipo |	Método |	Acción |
| -----|-----------|--------|
| Archivos idénticos, aunque tengan nombres diferentes	| SHA-256 del archivo	| Registrar las copias y conservar un representante, después de revisar etiquetas |
| Imágenes casi idénticas: otra compresión, tamaño o pequeños cambios |	Hash perceptual, pHash |	Revisar visualmente antes de excluir |


In [1]:
%pip install pillow ImageHash pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 6.6 MB/s eta 0:00:00


### Inventario de hashes

El pHash utilizado tiene 64 bits. La orientación se ajusta mediante los metadatos EXIF antes de calcularlo, para que esa información no altere innecesariamente la comparación.

In [4]:
from pathlib import Path
from PIL import Image, ImageOps
import hashlib
import imagehash
import pandas as pd

ROOT = Path("/content/plague-identification-model/data/raw")
REPORTS = Path("reports")
REPORTS.mkdir(parents=True, exist_ok=True)

extensions = {".jpg", ".jpeg", ".png"}
paths = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
)

if not paths:
    raise ValueError(f"No se encontraron imágenes en {ROOT}")

records = []

for path in paths:
    record = {
        "image_path": str(path),
        "sha256": None,
        "phash": None,
        "width": None,
        "height": None,
        "error": None,
    }

    try:
        sha = hashlib.sha256()

        with path.open("rb") as file:
            for block in iter(lambda: file.read(1024 * 1024), b""):
                sha.update(block)

        record["sha256"] = sha.hexdigest()

        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image.load()

            record["width"], record["height"] = image.size
            record["phash"] = str(imagehash.phash(image, hash_size=8))

    except Exception as exc:
        record["error"] = str(exc)

    records.append(record)

hash_df = pd.DataFrame(records)

hash_df.to_csv(REPORTS / "image_hashes.csv", index=False)

print("Imágenes encontradas:", len(hash_df))
print("Imágenes con errores:", hash_df["error"].notna().sum())

Imágenes encontradas: 6338
Imágenes con errores: 0


 ### Detectar duplicados exactos

 Agrupamos archivos con el mismo SHA-256:

In [5]:
valid_df = hash_df[hash_df["error"].isna()].copy()

exact_duplicates = valid_df[
    valid_df.duplicated("sha256", keep=False)
].copy()

exact_duplicates = exact_duplicates.sort_values(
    ["sha256", "image_path"]
)

exact_duplicates.to_csv(
    REPORTS / "exact_duplicates.csv",
    index=False
)

n_groups = exact_duplicates["sha256"].nunique()
n_extra_copies = len(exact_duplicates) - n_groups

print("Grupos de duplicados exactos:", n_groups)
print("Archivos involucrados:", len(exact_duplicates))
print("Copias adicionales:", n_extra_copies)

Grupos de duplicados exactos: 2
Archivos involucrados: 4
Copias adicionales: 2


Las duplicados detectados son de la misma etiqueta, por lo que no existen conflictos por resolver. Se optó por conservar la primera ruta en orden alfabético.

In [6]:
exact_sorted = exact_duplicates.sort_values(
    ["sha256", "image_path"]
).copy()

exact_sorted["keep_path"] = (
    exact_sorted.groupby("sha256")["image_path"]
    .transform("first")
)

excluded_exact = exact_sorted[
    exact_sorted["image_path"] != exact_sorted["keep_path"]
].copy()

excluded_exact["reason"] = "exact_duplicate"

excluded_exact[
    ["image_path", "keep_path", "sha256", "reason"]
].to_csv(
    REPORTS / "excluded_exact_duplicates.csv",
    index=False
)

display(excluded_exact[["image_path", "keep_path", "reason"]])

,image_path,keep_path,reason
1335,/content/plague-identification-model/data/raw/...,/content/plague-identification-model/data/raw/...,exact_duplicate
1518,/content/plague-identification-model/data/raw/...,/content/plague-identification-model/data/raw/...,exact_duplicate


### Detectar candidatos visualmente similares



In [7]:
representatives = (
    valid_df
    .sort_values("image_path")
    .drop_duplicates("sha256")
    .reset_index(drop=True)
)

items = [
    (row.image_path, int(row.phash, 16))
    for row in representatives.itertuples(index=False)
]

THRESHOLD = 5
similar_pairs = []

for i, (path_a, hash_a) in enumerate(items):
    for path_b, hash_b in items[i + 1:]:
        distance = (hash_a ^ hash_b).bit_count()

        if distance <= THRESHOLD:
            similar_pairs.append({
                "image_a": path_a,
                "image_b": path_b,
                "phash_distance": distance,
                "decision": "pending",
                "exclude_path": "",
                "notes": "",
            })

similar_df = pd.DataFrame(
    similar_pairs,
    columns=[
        "image_a",
        "image_b",
        "phash_distance",
        "decision",
        "exclude_path",
        "notes",
    ]
)

similar_df = similar_df.sort_values("phash_distance")

similar_df.to_csv(
    REPORTS / "similar_candidates.csv",
    index=False
)

print("Pares candidatos:", len(similar_df))

Pares candidatos: 1282
